<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Build step 5</div>
<div style="font-size:24px;font-weight:700;">&#9989; Validate the build</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">Checks every demo scenario the walkthrough relies on and prints a data fingerprint that is identical on every rebuild. A failed check fails the job, and so the bundle deploy.</p>


In [0]:
# Parameters come from the bundle's build job: the catalog and the three schemas the bundle created.
# This sets USE CATALOG and the session variables bronze_schema, silver_schema and gold_schema.
from bean_and_brew import init, fill

cfg = init(spark, dbutils)


In [0]:
# Every query below names the bundle's schemas through placeholders filled in by sql()
def sql(q):
    return spark.sql(fill(q, cfg))

EXPECTED_FINGERPRINT = 'ab02b95e172468b55ac67d170273121b11f8e795348cd90d4f8ac547a17ba1bc'

# Validation: every scenario the walkthrough relies on, plus a fingerprint that is identical on every rebuild
def one(q):
    return sql(q).first()

checks = []
def check(name, ok, detail):
    checks.append((("PASS" if ok else "FAIL"), name, str(detail)))

b = one("SELECT count(*) n, count(*) - count(DISTINCT transaction_id) dups, count_if(store_id <> upper(store_id)) lower FROM __BRONZE__.raw_pos_transactions")
s = one("SELECT count(*) n, count(*) - count(DISTINCT transaction_id) dups, count_if(store_id <> upper(store_id)) lower FROM __SILVER__.fact_transactions")
check("Bronze has duplicate and lower-case defects", b.dups > 0 and b.lower > 0, f"{b.n:,} rows, {b.dups:,} duplicate IDs, {b.lower:,} lower-case IDs")
check("Silver defects cleaned", s.dups == 0 and s.lower == 0, f"{s.n:,} tickets, {s.dups} duplicates, {s.lower} lower-case")

rec = sql("""
  SELECT (SELECT sum(net_sales) FROM __SILVER__.fact_transactions WHERE NOT is_voided) silver,
         (SELECT sum(net_sales) FROM __GOLD__.store_daily_performance) store_day,
         (SELECT sum(net_sales) FROM __GOLD__.product_sales_daily) product_day,
         (SELECT sum(net_sales) FROM __GOLD__.store_hourly_traffic) store_hour""").first()
check("Net sales reconcile across layers", len({rec.silver, rec.store_day, rec.product_day, rec.store_hour}) == 1, f"${rec.silver:,.2f}")

t = one("""SELECT avg(CASE WHEN business_date >= DATE'2026-10-15' THEN transactions END) /
                  avg(CASE WHEN business_date BETWEEN DATE'2026-09-15' AND DATE'2026-10-14' THEN transactions END) - 1 AS chg
           FROM __GOLD__.store_daily_performance WHERE store_id = 'S007'""")
check("(a) S007 traffic drops from 2026-10-15", t.chg < -0.4, f"{t.chg:+.1%} vs prior 30 days")

inv = [r.k for r in sql("SELECT concat(store_id, ' ', item_name) k FROM __GOLD__.inventory_current_status WHERE stock_status = 'REORDER NOW' ORDER BY 1").collect()]
check("(b) Exactly 3 items to reorder", inv == ["S004 Oat Milk", "S008 Oat Milk", "S010 Espresso Beans"], ", ".join(inv))

p = sql("""SELECT CASE WHEN business_date >= DATE'2026-01-05' THEN 'after' ELSE 'before' END period, MEASURE(avg_ticket) t
                 FROM __GOLD__.store_sales_metrics WHERE is_comp_store AND business_date BETWEEN DATE'2025-10-05' AND DATE'2026-04-05' GROUP BY ALL""").collect()
p = {r.period: float(r.t) for r in p}
check("(c) Average ticket rises after the 2026-01-05 price increase", p["after"] > p["before"], f"${p['before']:.2f} -> ${p['after']:.2f}")

n = one("SELECT min(business_date) first_day, count_if(is_comp_store) comp FROM __GOLD__.store_daily_performance WHERE store_id = 'S012'")
check("(d) RiNo opened 2026-03-01, no comp days yet", str(n.first_day) == "2026-03-01" and n.comp == 0, f"first day {n.first_day}, {n.comp} comp days")

c = one("""SELECT avg(CASE WHEN month(business_date) BETWEEN 5 AND 8 THEN u END) / avg(CASE WHEN month(business_date) NOT BETWEEN 5 AND 8 THEN u END) ratio
           FROM (SELECT date_trunc('MONTH', business_date) business_date, sum(units_sold) u FROM __GOLD__.product_sales_daily
                 WHERE category = 'Cold Drinks' AND year(business_date) = 2025 GROUP BY 1)""")
check("(e) Cold drinks peak May-Aug", c.ratio > 3, f"{c.ratio:.1f}x the rest of 2025")

m = sorted({r.m for r in sql("SELECT DISTINCT month(business_date) m FROM __GOLD__.product_sales_daily WHERE product_name = 'Pumpkin Spice Latte'").collect()})
check("(f) Pumpkin Spice Latte only Sep-Nov", set(m) <= {9, 10, 11} and m, f"months {m}")

labor = [r.store_id for r in sql("""SELECT store_id FROM __GOLD__.store_daily_performance WHERE business_date BETWEEN DATE'2026-09-22' AND DATE'2026-10-19'
                                          GROUP BY ALL HAVING sum(labor_cost) / sum(net_sales) > 0.30""").collect()]
check("Labor alert has a store over 30%", len(labor) > 0, ", ".join(labor))

promo = one("SELECT count(*) n FROM __GOLD__.loyalty_promo_targets")
check("Promo target list built", promo.n > 0, f"{promo.n:,} members")

fp = one("""SELECT sha2(concat_ws('|',
    (SELECT array_join(sort_array(collect_list(concat_ws(',', store_id, business_date, transactions, net_sales, labor_cost))), ';') FROM __GOLD__.store_daily_performance),
    (SELECT array_join(sort_array(collect_list(concat_ws(',', member_id, visits, lifetime_net_sales, favorite_product, customer_segment))), ';') FROM __GOLD__.customer_360),
    (SELECT array_join(sort_array(collect_list(concat_ws(',', store_id, item_code, on_hand_qty, stock_status))), ';') FROM __GOLD__.inventory_current_status),
    (SELECT cast(count(*) AS STRING) FROM __BRONZE__.raw_pos_transactions)), 256) fp""").fp

displayHTML("<table style='font-family:sans-serif;font-size:13px;border-collapse:collapse;'>" + "".join(
    f"<tr><td style='padding:3px 10px;color:{'#2E7D32' if r == 'PASS' else '#C62828'};font-weight:700;'>{r}</td><td style='padding:3px 10px;'>{n}</td><td style='padding:3px 10px;color:#5D4037;'>{d}</td></tr>"
    for r, n, d in checks) + f"</table><p style='font-family:monospace;font-size:12px;color:#5D4037;'>Data fingerprint: {fp}<br/>Reference build: {EXPECTED_FINGERPRINT}</p>")
failed = [n for r, n, _ in checks if r == "FAIL"]
assert not failed, f"Validation failed: {failed}"
if fp != EXPECTED_FINGERPRINT:
    print("NOTE: the fingerprint differs from the reference build, so the build SQL above was probably edited.")